In [3]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

OUTPUT_DIR   = Path.home() / "thermoelectric_dataset" / "output"
ANALYSIS_DIR = Path.home() / "thermoelectric_dataset" / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

df_full = pd.read_csv(OUTPUT_DIR / "C:\\Users\\Puneetha\\thermoelectric_dataset\\output\\thermoelectric_full.csv", low_memory=False)
df_ml   = pd.read_csv(OUTPUT_DIR / "C:\\Users\\Puneetha\\thermoelectric_dataset\\output\\thermoelectric_ml_ready.csv", low_memory=False)

print(f"Full dataset:    {df_full.shape[0]:,} rows x {df_full.shape[1]} columns")
print(f"ML-ready subset: {df_ml.shape[0]:,} rows x {df_ml.shape[1]} columns")

Full dataset:    77,108 rows x 57 columns
ML-ready subset: 24,024 rows x 57 columns


In [4]:
df_full.head()

,jid,formula,spg_symbol,spg_number,crys,dimensionality,_dim,_source_db,n-Seebeck,p-Seebeck,...,spillage,efg,max_efg,density,nat,Tc_supercon,atoms,icsd,edos_up,pdos_elast
0,JVASP-90856,NaN,P4/nmm,129,tetragonal,3D-bulk,3D,dft_3d,NaN,NaN,...,NaN,[],NaN,5.956,8,NaN,"{'lattice_mat': [[3.566933224304235, 0.0, -0.0...",NaN,"[0.026399112342704156, 0.026668810423400677, 0...",na
1,JVASP-86097,NaN,Pm-3m,221,cubic,3D-bulk,3D,dft_3d,NaN,NaN,...,NaN,[],NaN,5.522,7,NaN,"{'lattice_mat': [[4.089078911208881, 0.0, 0.0]...",NaN,"[0.03475311229090909, 0.03548667718787879, 0.0...",na
2,JVASP-64906,NaN,I-4m2,119,tetragonal,intercalated ion,3D,dft_3d,NaN,NaN,...,NaN,"[['Be', 'a', '-1.504', '0.001', '0.001', '0.00...",23.940,10.960,4,NaN,"{'lattice_mat': [[-1.833590720595598, 1.833590...",NaN,"[0.047606193133047224, 0.047140801323319034, 0...",na
3,JVASP-98225,NaN,P2_1/c,14,monoclinic,intercalated ion,3D,dft_3d,NaN,NaN,...,NaN,[],NaN,5.145,32,NaN,"{'lattice_mat': [[7.2963518353359165, 0.0, 0.0...",55065.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",na
4,JVASP-10,NaN,P-3m1,164,trigonal,2D-bulk,3D,dft_3d,18.273333,17.486667,...,NaN,"[['V', 'a', '22.568', '0.0', '0.0', '0.0', '22...",89.678,5.718,3,NaN,"{'lattice_mat': [[1.6777483798834445, -2.90594...",NaN,"[0.03708017076086954, 0.04270037632447661, 0.0...","[5.756737314030563e-06, 2.4058545739267754e-05..."


In [5]:
df_full.columns

Index(['jid', 'formula', 'spg_symbol', 'spg_number', 'crys', 'dimensionality',
       '_dim', '_source_db', 'n-Seebeck', 'p-Seebeck', 'n-powerfact',
       'p-powerfact', 'nkappa', 'pkappa', 'ncond', 'pcond',
       'optb88vdw_bandgap', 'mbj_bandgap', 'hse_gap', 'avg_elec_mass',
       'avg_hole_mass', 'effective_masses_300K', 'formation_energy_peratom',
       'ehull', 'optb88vdw_total_energy', 'exfoliation_energy',
       'bulk_modulus_kv', 'shear_modulus_gv', 'elastic_tensor', 'poisson',
       'modes', 'max_ir_mode', 'min_ir_mode', 'epsx', 'epsy', 'epsz', 'mepsx',
       'mepsy', 'mepsz', 'dfpt_piezo_max_dielectric',
       'dfpt_piezo_max_dielectric_electronic',
       'dfpt_piezo_max_dielectric_ionic', 'dfpt_piezo_max_eij',
       'dfpt_piezo_max_dij', 'magmom_oszicar', 'magmom_outcar', 'slme',
       'spillage', 'efg', 'max_efg', 'density', 'nat', 'Tc_supercon', 'atoms',
       'icsd', 'edos_up', 'pdos_elast'],
      dtype='object')

In [6]:
# Cell 2 — Missing value count and % for every column
def missing_report(df, name):
    total   = len(df)
    missing = df.isnull().sum()
    pct     = (missing / total * 100).round(2)
    report  = pd.DataFrame({
        "missing_count": missing,
        "missing_%":     pct,
        "present_count": total - missing,
        "present_%":     (100 - pct).round(2)
    }).sort_values("missing_%", ascending=False)
    print(f"\n{'='*55}")
    print(f"Missing Data Report — {name} ({total:,} materials)")
    print(f"{'='*55}")
    print(report.to_string())
    return report

report_full = missing_report(df_full, "Full Dataset")
report_ml   = missing_report(df_ml,   "ML-Ready Subset")


Missing Data Report — Full Dataset (77,108 materials)
                                      missing_count  missing_%  present_count  present_%
hse_gap                                       76998      99.86            110       0.14
formula                                       76003      98.57           1105       1.43
exfoliation_energy                            75547      97.98           1561       2.02
Tc_supercon                                   74947      97.20           2161       2.80
dfpt_piezo_max_dij                            72658      94.23           4450       5.77
dfpt_piezo_max_dielectric                     72402      93.90           4706       6.10
dfpt_piezo_max_eij                            72309      93.78           4799       6.22
max_ir_mode                                   72303      93.77           4805       6.23
dfpt_piezo_max_dielectric_electronic          72299      93.76           4809       6.24
min_ir_mode                                   72299    

In [8]:
no_null_cols = df_ml.columns[df_ml.isnull().sum() == 0].tolist()
print(f"Columns with 0 null values ({len(no_null_cols)} / {df_ml.shape[1]}):\n")
for col in no_null_cols:
    print(f"  - {col}  ({df_ml[col].dtype})")

Columns with 0 null values (19 / 57):

  - jid  (object)
  - spg_symbol  (object)
  - spg_number  (int64)
  - crys  (object)
  - dimensionality  (object)
  - _dim  (object)
  - _source_db  (object)
  - n-Seebeck  (float64)
  - optb88vdw_bandgap  (float64)
  - effective_masses_300K  (object)
  - formation_energy_peratom  (float64)
  - ehull  (float64)
  - optb88vdw_total_energy  (float64)
  - elastic_tensor  (object)
  - modes  (object)
  - efg  (object)
  - density  (float64)
  - nat  (int64)
  - atoms  (object)


In [7]:
no_null_cols = df_full.columns[df_full.isnull().sum() == 0].tolist()
print(f"Columns with 0 null values ({len(no_null_cols)} / {df_full.shape[1]}):\n")
for col in no_null_cols:
    print(f"  - {col}  ({df_full[col].dtype})")

Columns with 0 null values (18 / 57):

  - jid  (object)
  - spg_symbol  (object)
  - spg_number  (int64)
  - crys  (object)
  - dimensionality  (object)
  - _dim  (object)
  - _source_db  (object)
  - optb88vdw_bandgap  (float64)
  - effective_masses_300K  (object)
  - formation_energy_peratom  (float64)
  - ehull  (float64)
  - optb88vdw_total_energy  (float64)
  - elastic_tensor  (object)
  - modes  (object)
  - efg  (object)
  - density  (float64)
  - nat  (int64)
  - atoms  (object)


In [ ]:
threshold = 0.75
total = len(df_full)
non_null_frac = df_full.notnull().sum() / total

cols_75 = non_null_frac[non_null_frac > threshold].sort_values(ascending=False)
print(f"Columns with >{threshold*100:.0f}% non-null values ({len(cols_75)} / {df_full.shape[1]}):\n")
for col, frac in cols_75.items():
    print(f"  {col:45s}  {frac*100:6.2f}%  ({df_full[col].notnull().sum():,} / {total:,})")